# Feature Space Alignment: IRSE50 vs InsightFace ArcFace R50

驗證兩個模型的 512 維 embedding 特徵空間有多相似，決定要不要寫 Mapper。

**獨立 notebook** — 不需要任何前置條件，從零跑到底。

**流程：**
1. 安裝套件 + 寫 irse.py + 下載 irse50.pth → **自動重啟 kernel**
2. 載入 IRSE50 + InsightFace ArcFace R50
3. 下載 LFW deep-funneled（~110MB）
4. 抽 N=500 張臉的兩組 embedding
5. Spearman correlation + Procrustes R²
6. 自動結論

**判讀標準：**
- Spearman > 0.9 + R² > 0.85 → 不用 mapper
- 0.7~0.9, 0.6~0.85 → mapper 有用 (預期 transfer +10~20%)
- < 0.7 → mapper 是必要的

## 1. 安裝套件 + 設置（會自動重啟 kernel）

跑完這格 Colab 會 crash 重啟（這是預期行為）。重啟後從 cell 4 開始往下跑，不要再跑這格。

In [ ]:
import os, sys, subprocess
from pathlib import Path

# 安裝必要套件（--no-deps 避免 numpy 被升到 2.x）
!python -m pip install -q --no-deps insightface==0.7.3 onnxruntime-gpu==1.19.2
!python -m pip install -q --no-deps onnx==1.16.1 easydict
!python -m pip install -q --force-reinstall --no-cache-dir 'numpy==1.26.4'
!python -m pip install -q 'protobuf>=4.25,<5' scikit-learn matplotlib scipy tqdm

# 寫 irse.py
assets_dir = Path('/content/assets/models')
assets_dir.mkdir(parents=True, exist_ok=True)
(assets_dir / '__init__.py').write_text('')

IRSE_PY = r"""
from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, ReLU, Sigmoid, Dropout2d, Dropout, AvgPool2d, MaxPool2d, AdaptiveAvgPool2d, Sequential, Module, Parameter
import torch.nn.functional as F
import torch
from collections import namedtuple
import math
import pdb

##################################  Original Arcface Model #############################################################

class Flatten(Module):
    def forward(self, input):
        return input.reshape(input.size(0), -1)

def l2_norm(input,axis=1):
    norm = torch.norm(input,2,axis,True)
    output = torch.div(input, norm)
    return output

class SEModule(Module):
    def __init__(self, channels, reduction):
        super(SEModule, self).__init__()
        self.avg_pool = AdaptiveAvgPool2d(1)
        self.fc1 = Conv2d(
            channels, channels // reduction, kernel_size=1, padding=0 ,bias=False)
        self.relu = ReLU(inplace=True)
        self.fc2 = Conv2d(
            channels // reduction, channels, kernel_size=1, padding=0 ,bias=False)
        self.sigmoid = Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.avg_pool(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return module_input * x

class bottleneck_IR(Module):
    def __init__(self, in_channel, depth, stride):
        super(bottleneck_IR, self).__init__()
        if in_channel == depth:
            self.shortcut_layer = MaxPool2d(1, stride)
        else:
            self.shortcut_layer = Sequential(
                Conv2d(in_channel, depth, (1, 1), stride ,bias=False), BatchNorm2d(depth))
        self.res_layer = Sequential(
            BatchNorm2d(in_channel),
            Conv2d(in_channel, depth, (3, 3), (1, 1), 1 ,bias=False), PReLU(depth),
            Conv2d(depth, depth, (3, 3), stride, 1 ,bias=False), BatchNorm2d(depth))

    def forward(self, x):
        shortcut = self.shortcut_layer(x)
        res = self.res_layer(x)
        return res + shortcut

class bottleneck_IR_SE(Module):
    def __init__(self, in_channel, depth, stride):
        super(bottleneck_IR_SE, self).__init__()
        if in_channel == depth:
            self.shortcut_layer = MaxPool2d(1, stride)
        else:
            self.shortcut_layer = Sequential(
                Conv2d(in_channel, depth, (1, 1), stride ,bias=False), 
                BatchNorm2d(depth))
        self.res_layer = Sequential(
            BatchNorm2d(in_channel),
            Conv2d(in_channel, depth, (3,3), (1,1),1 ,bias=False),
            PReLU(depth),
            Conv2d(depth, depth, (3,3), stride, 1 ,bias=False),
            BatchNorm2d(depth),
            SEModule(depth,16)
            )
    def forward(self,x):
        shortcut = self.shortcut_layer(x)
        res = self.res_layer(x)
        return res + shortcut

class Bottleneck(namedtuple('Block', ['in_channel', 'depth', 'stride'])):
    '''A named tuple describing a ResNet block.'''
    
def get_block(in_channel, depth, num_units, stride = 2):
  return [Bottleneck(in_channel, depth, stride)] + [Bottleneck(depth, depth, 1) for i in range(num_units-1)]

def get_blocks(num_layers):
    if num_layers == 50:
        blocks = [
            get_block(in_channel=64, depth=64, num_units = 3),
            get_block(in_channel=64, depth=128, num_units=4),
            get_block(in_channel=128, depth=256, num_units=14),
            get_block(in_channel=256, depth=512, num_units=3)
        ]
    elif num_layers == 100:
        blocks = [
            get_block(in_channel=64, depth=64, num_units=3),
            get_block(in_channel=64, depth=128, num_units=13),
            get_block(in_channel=128, depth=256, num_units=30),
            get_block(in_channel=256, depth=512, num_units=3)
        ]
    elif num_layers == 152:
        blocks = [
            get_block(in_channel=64, depth=64, num_units=3),
            get_block(in_channel=64, depth=128, num_units=8),
            get_block(in_channel=128, depth=256, num_units=36),
            get_block(in_channel=256, depth=512, num_units=3)
        ]
    return blocks

class Backbone(Module):
    def __init__(self, num_layers, drop_ratio, mode='ir'):
        super(Backbone, self).__init__()
        assert num_layers in [50, 100, 152], 'num_layers should be 50,100, or 152'
        assert mode in ['ir', 'ir_se'], 'mode should be ir or ir_se'
        blocks = get_blocks(num_layers)
        if mode == 'ir':
            unit_module = bottleneck_IR
        elif mode == 'ir_se':
            unit_module = bottleneck_IR_SE
        self.input_layer = Sequential(Conv2d(3, 64, (3, 3), 1, 1 ,bias=False), 
                                      BatchNorm2d(64), 
                                      PReLU(64))
        self.output_layer = Sequential(BatchNorm2d(512), 
                                       Dropout(drop_ratio),
                                       Flatten(),
                                       Linear(512 * 7 * 7, 512),
                                       BatchNorm1d(512))
        modules = []
        for block in blocks:
            for bottleneck in block:
                modules.append(
                    unit_module(bottleneck.in_channel,
                                bottleneck.depth,
                                bottleneck.stride))
        self.body = Sequential(*modules)
    
    def forward(self,x):
        x = self.input_layer(x)
        x = self.body(x)
        x = self.output_layer(x)
        return l2_norm(x)

##################################  MobileFaceNet #############################################################
    
class Conv_block(Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Conv_block, self).__init__()
        self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding, bias=False)
        self.bn = BatchNorm2d(out_c)
        self.prelu = PReLU(out_c)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.prelu(x)
        return x

class Linear_block(Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Linear_block, self).__init__()
        self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding, bias=False)
        self.bn = BatchNorm2d(out_c)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x

class Depth_Wise(Module):
     def __init__(self, in_c, out_c, residual = False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1):
        super(Depth_Wise, self).__init__()
        self.conv = Conv_block(in_c, out_c=groups, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.conv_dw = Conv_block(groups, groups, groups=groups, kernel=kernel, padding=padding, stride=stride)
        self.project = Linear_block(groups, out_c, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.residual = residual
     def forward(self, x):
        if self.residual:
            short_cut = x
        x = self.conv(x)
        x = self.conv_dw(x)
        x = self.project(x)
        if self.residual:
            output = short_cut + x
        else:
            output = x
        return output

class Residual(Module):
    def __init__(self, c, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1)):
        super(Residual, self).__init__()
        modules = []
        for _ in range(num_block):
            modules.append(Depth_Wise(c, c, residual=True, kernel=kernel, padding=padding, stride=stride, groups=groups))
        self.model = Sequential(*modules)
    def forward(self, x):
        return self.model(x)

class MobileFaceNet(Module):
    def __init__(self, embedding_size):
        super(MobileFaceNet, self).__init__()
        self.conv1 = Conv_block(3, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1))
        self.conv2_dw = Conv_block(64, 64, kernel=(3, 3), stride=(1, 1), padding=(1, 1), groups=64)
        self.conv_23 = Depth_Wise(64, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=128)
        self.conv_3 = Residual(64, num_block=4, groups=128, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_34 = Depth_Wise(64, 128, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
        self.conv_4 = Residual(128, num_block=6, groups=256, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_45 = Depth_Wise(128, 128, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=512)
        self.conv_5 = Residual(128, num_block=2, groups=256, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_6_sep = Conv_block(128, 512, kernel=(1, 1), stride=(1, 1), padding=(0, 0))
        self.conv_6_dw = Linear_block(512, 512, groups=512, kernel=(7,7), stride=(1, 1), padding=(0, 0))
        self.conv_6_flatten = Flatten()
        self.linear = Linear(512, embedding_size, bias=False)
        self.bn = BatchNorm1d(embedding_size)
    
    def forward(self, x):
        out = self.conv1(x)

        out = self.conv2_dw(out)

        out = self.conv_23(out)

        out = self.conv_3(out)
        
        out = self.conv_34(out)

        out = self.conv_4(out)

        out = self.conv_45(out)

        out = self.conv_5(out)

        out = self.conv_6_sep(out)

        out = self.conv_6_dw(out)

        out = self.conv_6_flatten(out)

        out = self.linear(out)

        out = self.bn(out)
        return l2_norm(out)

##################################  Arcface head #############################################################


"""
(assets_dir / 'irse.py').write_text(IRSE_PY)
print('寫 /content/assets/models/irse.py')

# 下載 irse50.pth（與 DiffProtect cell 8 同一個 gdrive zip）
if not (assets_dir / 'irse50.pth').exists():
    print('Downloading irse50.pth from DiffProtect assets zip...')
    os.system("rm -rf /content/diffprotect_assets_extract && mkdir -p /content/diffprotect_assets_extract")
    os.system("gdown --fuzzy 'https://drive.google.com/file/d/19_Y0jR789BGciogjjoGtWNEv-5QBiCB7/view?usp=sharing' -O /content/diffprotect_assets.zip")
    os.system("unzip -q -o /content/diffprotect_assets.zip -d /content/diffprotect_assets_extract")
    # 把 irse50.pth 撈出來
    import glob, shutil
    for f in glob.glob('/content/diffprotect_assets_extract/**/irse50.pth', recursive=True):
        shutil.copy2(f, assets_dir / 'irse50.pth')
        print(f'Copied {f} -> {assets_dir / "irse50.pth"}')
        break

print('irse50.pth 存在:', (assets_dir / 'irse50.pth').exists())

print('\n環境設置完成，自動重啟 Colab runtime 以套用 numpy 1.26.4...')
print('重啟後請從下面的 cell 開始跑（不要再跑這格）')
os.kill(os.getpid(), 9)

## 2. 載入兩個模型

⚠️ **這格是 runtime 重啟之後的起點**。如果你看到 "Your session crashed" 訊息，那是上一格主動觸發的，**請從這格繼續往下跑**。

In [ ]:
import os, sys, types
from pathlib import Path
import numpy as np

assert np.__version__.startswith('1.26'), f'numpy 版本錯了: {np.__version__}'
print('numpy:', np.__version__)

# Stub albumentations（避免 numpy 1.26 + alb 1.4 的 KeyError）
for name in list(sys.modules):
    if name.split('.')[0] in {'insightface', 'albumentations'}:
        del sys.modules[name]

_alb = types.ModuleType('albumentations'); _alb.__path__ = []
_alb_core = types.ModuleType('albumentations.core'); _alb_core.__path__ = []
_alb_ti = types.ModuleType('albumentations.core.transforms_interface')
class _Stub:
    def __init__(self, *a, **k): pass
    def __call__(self, **kw): return kw
_alb_ti.ImageOnlyTransform = _Stub; _alb_ti.DualTransform = _Stub
_alb_aug = types.ModuleType('albumentations.augmentations'); _alb_aug.__path__ = []
_alb_pt = types.ModuleType('albumentations.pytorch'); _alb_pt.__path__ = []
sys.modules.update({
    'albumentations': _alb,
    'albumentations.core': _alb_core,
    'albumentations.core.transforms_interface': _alb_ti,
    'albumentations.augmentations': _alb_aug,
    'albumentations.pytorch': _alb_pt,
})
_alb.core = _alb_core; _alb_core.transforms_interface = _alb_ti
_alb.augmentations = _alb_aug; _alb.pytorch = _alb_pt
_alb.ImageOnlyTransform = _Stub

import torch, cv2
from insightface.app import FaceAnalysis
from insightface.utils import face_align

INSIGHT_APP = FaceAnalysis(name='buffalo_l',
                           providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
INSIGHT_APP.prepare(ctx_id=0, det_size=(640, 640))
INSIGHT_REC = INSIGHT_APP.models['recognition']
print('InsightFace recognition input:', INSIGHT_REC.input_shape)

sys.path.insert(0, '/content')
from assets.models import irse
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
irse50_model = irse.Backbone(50, 0.6, 'ir_se').to(device)
irse50_model.load_state_dict(torch.load('/content/assets/models/irse50.pth',
                                         map_location=device, weights_only=False))
irse50_model.eval()
print('IRSE50 loaded on', device)

## 3. 下載 LFW deep-funneled 子集

In [ ]:
import os, tarfile, urllib.request, glob, random
from pathlib import Path

LFW_DIR = Path('/content/lfw-deepfunneled')
LFW_TGZ = Path('/content/lfw-deepfunneled.tgz')

if not LFW_DIR.exists():
    if not LFW_TGZ.exists():
        print('Downloading LFW deep-funneled (~110MB)...')
        urllib.request.urlretrieve(
            'http://vis-www.cs.umass.edu/lfw/lfw-deepfunneled.tgz',
            str(LFW_TGZ)
        )
    print('Extracting...')
    with tarfile.open(LFW_TGZ) as t:
        t.extractall('/content/')
    print('Done')
else:
    print('LFW already extracted')

random.seed(42)
all_paths = sorted(glob.glob('/content/lfw-deepfunneled/*/*.jpg'))
print(f'Total LFW images: {len(all_paths)}')
sample_paths = random.sample(all_paths, 500)
print(f'Sampled: {len(sample_paths)}')

## 4. 抽 embedding（兩個模型各跑一次）

In [ ]:
from tqdm import tqdm

def get_aligned_112(bgr_img, app):
    faces = app.get(bgr_img)
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    aligned = face_align.norm_crop(bgr_img, landmark=face.kps, image_size=112)
    return aligned

def emb_insightface(aligned_bgr):
    emb = INSIGHT_REC.get_feat(aligned_bgr)
    if emb.ndim == 2: emb = emb[0]
    return emb.astype(np.float32)

def emb_irse(aligned_bgr):
    rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - 0.5) / 0.5
    t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = irse50_model(t).cpu().numpy()[0]
    return emb.astype(np.float32)

def l2_normalize(v):
    return v / (np.linalg.norm(v) + 1e-12)

E_INSIGHT, E_IRSE = [], []
skipped = 0
for p in tqdm(sample_paths):
    img = cv2.imread(str(p))
    if img is None: skipped += 1; continue
    aligned = get_aligned_112(img, INSIGHT_APP)
    if aligned is None: skipped += 1; continue
    e_in = l2_normalize(emb_insightface(aligned))
    e_ir = l2_normalize(emb_irse(aligned))
    E_INSIGHT.append(e_in)
    E_IRSE.append(e_ir)

E_INSIGHT = np.stack(E_INSIGHT)
E_IRSE = np.stack(E_IRSE)
print(f'N: {len(E_INSIGHT)}, skipped: {skipped}')
print(f'E_INSIGHT shape: {E_INSIGHT.shape}')
print(f'E_IRSE shape:    {E_IRSE.shape}')

## 5. Spearman 相關 + 散布圖

In [ ]:
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt

C_INSIGHT = E_INSIGHT @ E_INSIGHT.T
C_IRSE    = E_IRSE @ E_IRSE.T

N = len(E_INSIGHT)
iu = np.triu_indices(N, k=1)
v_insight = C_INSIGHT[iu]
v_irse    = C_IRSE[iu]

spearman_r, _ = spearmanr(v_insight, v_irse)
pearson_r, _  = pearsonr(v_insight, v_irse)

print(f'Pairwise cosine sim (n_pairs = {len(v_insight)})')
print(f'  Spearman: {spearman_r:.4f}')
print(f'  Pearson:  {pearson_r:.4f}')

plt.figure(figsize=(6, 6))
plt.scatter(v_insight, v_irse, s=2, alpha=0.3)
plt.plot([-0.5, 1], [-0.5, 1], 'r--', alpha=0.5, label='y=x')
plt.xlabel('cos sim (InsightFace ArcFace R50)')
plt.ylabel('cos sim (IRSE50)')
plt.title(f'Pairwise cosine sim correlation\nSpearman={spearman_r:.3f}, Pearson={pearson_r:.3f}')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 6. Linear Procrustes Alignment R²

In [ ]:
from scipy.linalg import orthogonal_procrustes

W, scale = orthogonal_procrustes(E_IRSE, E_INSIGHT)
print(f'Procrustes scale factor: {scale:.4f}')

E_IRSE_mapped = E_IRSE @ W

ss_res = ((E_INSIGHT - E_IRSE_mapped) ** 2).sum()
ss_tot = ((E_INSIGHT - E_INSIGHT.mean(0)) ** 2).sum()
r_squared = 1 - ss_res / ss_tot

per_img_cos_raw    = (E_INSIGHT * E_IRSE).sum(axis=1)
per_img_cos_mapped = (E_INSIGHT * E_IRSE_mapped).sum(axis=1) / (
    np.linalg.norm(E_INSIGHT, axis=1) * np.linalg.norm(E_IRSE_mapped, axis=1) + 1e-12)

print(f'Procrustes R²:                          {r_squared:.4f}')
print(f'Mean cos(mapped_irse, insight):         {per_img_cos_mapped.mean():.4f}')
print(f'Mean cos(raw_irse, insight) baseline:   {per_img_cos_raw.mean():.4f}')

C_IRSE_MAPPED = E_IRSE_mapped @ E_IRSE_mapped.T
v_irse_mapped = C_IRSE_MAPPED[iu]
spearman_after, _ = spearmanr(v_insight, v_irse_mapped)
pearson_after, _  = pearsonr(v_insight, v_irse_mapped)
print(f'\nAfter Procrustes:')
print(f'  Spearman: {spearman_after:.4f} (was {spearman_r:.4f})')
print(f'  Pearson:  {pearson_after:.4f} (was {pearson_r:.4f})')

## 7. 結論

In [ ]:
print('='*55)
print('  Feature Space Alignment 結論')
print('='*55)
print(f'IRSE50 vs InsightFace ArcFace R50 (N={len(E_INSIGHT)})')
print(f'  Spearman correlation:           {spearman_r:.4f}')
print(f'  Procrustes R²:                  {r_squared:.4f}')
print(f'  Cos(raw_irse, insight):         {per_img_cos_raw.mean():.4f}')
print(f'  Cos(mapped_irse, insight):      {per_img_cos_mapped.mean():.4f}')
print()

if spearman_r > 0.9 and r_squared > 0.85:
    verdict = '兩空間高度對齊 — 直接 transfer 應該就夠好，不用 mapper'
elif spearman_r > 0.7 and r_squared > 0.6:
    verdict = 'mapper 有用，預期 transfer attack 成功率 +10~20%'
else:
    verdict = 'mapper 是必要的，目前 transfer 很可能失敗'

print('  >>>', verdict)
print('='*55)